# Vinamilk embedding benchmark (full public corpus, cloud GPU)

Compare the proposal's three exact embedding models on all 4,750 public manifest passages and the same 12 role-labeled business cases. The private Kaggle dataset `ltrngnhn/vinamilk-proposal-benchmark-assets` contains the corpus, cases, and evaluator but no database credential or private uploads. Enable **Internet** and a **GPU accelerator**.

The evaluator applies each document's role filter before scoring. It runs each model in a fresh process with the same 512-token cap and batch size. This remains a smoke comparison: 12 document-level judgments are too few to select a production model.

In [ ]:
from pathlib import Path
import shutil

input_root = Path("/kaggle/input")
assets_candidates = sorted(
    project for project in input_root.rglob("project")
    if (project / "scripts" / "scripts.processing.evaluate_ocr.py").is_file()
    and (project / "scripts" / "scripts.retrieval.evaluate_embeddings.py").is_file()
)
if len(assets_candidates) != 1:
    raise FileNotFoundError(
        "Expected exactly one attached vinamilk-proposal-benchmark-assets/project "
        "containing scripts/retrieval/evaluate_embeddings.py; "
        f"found: {assets_candidates}"
    )
assets = assets_candidates[0]
repo = Path("/kaggle/working/enterprise-knowledge-ai-agent")
if repo.exists():
    shutil.rmtree(repo)
shutil.copytree(assets, repo)
%cd /kaggle/working/enterprise-knowledge-ai-agent

In [ ]:
import hashlib, json

corpus = Path("data/evaluation/vinamilk_public_embedding_corpus.jsonl")
cases = Path("data/evaluation/vinamilk_retrieval_cases.json")
assert corpus.is_file() and cases.is_file()
metadata = json.loads(Path(str(corpus) + ".metadata.json").read_text())
assert hashlib.sha256(corpus.read_bytes()).hexdigest() == metadata["corpus_sha256"]
print(metadata)


In [ ]:
import subprocess, sys

# Keep the embedding framework version aligned with the repository lockfile.
subprocess.run([sys.executable, "-m", "pip", "install", "sentence-transformers==6.0.1"], check=True)
import torch
print("torch", torch.__version__, "CUDA", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("GPU requested but CUDA is unavailable; restart with a Kaggle GPU accelerator.")

In [ ]:
import subprocess, sys

# Separate processes release model allocations between runs.
models = ["bge-m3", "qwen3-embedding", "e5-large-v2"]
results = Path("/kaggle/working/embedding-results")
results.mkdir(exist_ok=True)
for model in models:
    report = results / f"vinamilk-embedding-{model}-full-public-corpus.json"
    command = [sys.executable, "scripts/retrieval/evaluate_embeddings.py", "--model", model, "--corpus", str(corpus), "--cases", str(cases), "--device", "cuda", "--batch-size", "4", "--max-chunks-per-document", "1000", "--top-k", "30", "--output", str(report)]
    subprocess.run(command, check=True)
    print(f"Saved {report}")


## Result files

Download the three JSON reports from `/kaggle/working/embedding-results/`. Each includes model revision, corpus/case hashes, GPU name, peak CPU/GPU memory, timings, role-filtered rankings, and per-case metrics. Keep the raw reports in `artifacts/evaluation/`; do not replace the prior paired smoke reports because this run uses a larger corpus and different hardware.
